# Pipeline de Avaliação de Modelos de Detecção de Anomalias — Censo Agropecuário

Este notebook implementa um pipeline completo de detecção de anomalias para os dados do Censo Agropecuário 2017, utilizando modelos da biblioteca **PyOD**. O pipeline foi desenhado para dados não-paramétricos e esparsos, com taxa de contaminação estimada em **6,87%**.

## Fluxo geral

```
Dados brutos (parquet)
       │
       ▼
 Merge com ground truth (KEY_COLS)
       │
       ▼
 Amostragem estratificada (10k)
       │
       ▼
 Split treino/teste (80/20)
       │
       ▼
 Codificação (AnomalyDetectionEncoder)
       │
       ▼
 Imputação residual (SimpleImputer fill=0)
       │
       ├─── Experimento redução de dimensionalidade (Seção 4b)
       │
       ▼
 Treinamento e scoring (MODELS_REGISTRY)
       │
       ▼
 Métricas por modelo/dataset (ROC-AUC, AP, P@K, F1)
       │
       ▼
 Análise de concordância entre modelos
       │
       ▼
 Avaliação combinada multi-dataset (max score)
       │
       ▼
 Explicabilidade (SHAP — melhor modelo)
       │
       ▼
 Exportação de resultados
```

## Justificativa das escolhas

| Modelo | Família | Justificativa |
|--------|---------|---------------|
| IForest | Ensemble | Robusto a dados esparsos; eficiente para alta dimensionalidade; não assume distribuição |
| HBOS | Histograma | Extremamente rápido; funciona bem com features independentes; adequado para dados mistos |
| ECOD | Probabilístico | Sem parâmetros críticos; usa distribuição empírica cumulativa; eficaz em dados não-paramétricos |
| COPOD | Probabilístico | Similar ao ECOD; detecta outliers por cópulas empíricas; boa calibração de scores |
| LODA | Ensemble | Projeções aleatórias esparsas; especialmente adequado para dados esparsos e de alta dimensão |
| LOF | Proximidade | Detecta anomalias locais por densidade; complementa modelos globais |

## Métrica principal: Average Precision (AP)

Com apenas ~6,87% de positivos, a AP é mais informativa que ROC-AUC pois avalia a qualidade do ranking nas primeiras posições, onde os verdadeiros anomalias devem estar.

## Seção 0 — Setup e Configuração

In [1]:
import sys
import os
import warnings
import time
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona o diretório pai ao path para importar o encoder
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from encoder import AnomalyDetectionEncoder

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.loda import LODA
from pyod.models.lof import LOF
from pyod.models.kpca import KPCA

import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ─────────────────────────────────────────────────────────────
# Constantes globais — ajuste LABELS_PATH antes de executar
# ─────────────────────────────────────────────────────────────
KEY_COLS        = ['V010100', 'NUM_QUADRA', 'NUM_FACE', 'V010800']
CONTAMINATION   = 0.0687
SAMPLE_SIZE     = 10_000
RANDOM_STATE    = 42
LABELS_PATH     = '../data/ground_truth.csv'   # ← forneça o CSV com is_anomaly e KEY_COLS
DATA_DIR        = '../data/experimentos/abordagem1'
OUTPUT_DIR      = '../data/experimentos/abordagem1'
EXPLAIN_MODEL_NAME  = None   # None = auto (melhor AP); ou ex.: "IForest", "ECOD"
EXPLAIN_DATASET     = 'estabel'

print("Setup concluído.")
print(f"  CONTAMINATION = {CONTAMINATION}")
print(f"  SAMPLE_SIZE   = {SAMPLE_SIZE}")
print(f"  LABELS_PATH   = {LABELS_PATH}")

Hello, world!
Setup concluído.
  CONTAMINATION = 0.0687
  SAMPLE_SIZE   = 10000
  LABELS_PATH   = ../data/ground_truth.csv


## Seção 2 — Carregamento dos Labels (Ground Truth)

O arquivo de ground truth deve ser um CSV com as seguintes colunas:
- `V010100`, `NUM_QUADRA`, `NUM_FACE`, `V010800` — chave composta (KEY_COLS)
- `is_anomaly` — 0 ou 1 (obrigatório)
- `origem_anotacao` — rótulo textual da origem (opcional, ex.: `"regra:quadro02"`, `"anotacao_direta"`)

Estabelecimentos **não** presentes no CSV receberão `is_anomaly = 0`.

In [ ]:
if not os.path.exists(LABELS_PATH):
    raise FileNotFoundError(
        f"\n[ERRO] Arquivo de ground truth não encontrado: {LABELS_PATH}\n"
        f"Forneça um CSV com as colunas: {KEY_COLS + ['is_anomaly']} (+ 'origem_anotacao' opcional).\n"
        f"Atualize LABELS_PATH na Seção 0 antes de continuar."
    )

labels_df = pd.read_csv(LABELS_PATH)

# Validar colunas obrigatórias
required_cols = KEY_COLS + ['is_anomaly']
missing = [c for c in required_cols if c not in labels_df.columns]
if missing:
    raise ValueError(f"[ERRO] Colunas ausentes no arquivo de labels: {missing}")

# Garantir tipos corretos nas KEY_COLS
for col in KEY_COLS:
    labels_df[col] = labels_df[col].astype(str)

labels_df['is_anomaly'] = labels_df['is_anomaly'].astype(int)

print(f"Labels carregados: {len(labels_df):,} registros")
print("\nDistribuição de is_anomaly:")
print(labels_df['is_anomaly'].value_counts().rename({0: 'Normal (0)', 1: 'Anomalia (1)'}).to_string())
print(f"\nTaxa de anomalias: {labels_df['is_anomaly'].mean():.2%}")

if 'origem_anotacao' in labels_df.columns:
    print("\nDistribuição por origem_anotacao:")
    print(labels_df.groupby(['origem_anotacao', 'is_anomaly']).size().to_string())

## Seção 3 — Pré-processamento

Para cada dataset (estabel, lav_temp, pecu):
1. Carrega o parquet pós-feature-engineering
2. Garante que KEY_COLS existam como strings e faz merge com ground truth (left join → NaN → 0)
3. Amostragem estratificada por `is_anomaly` (até 10k registros)
4. Split 80/20 treino/teste estratificado
5. Ajusta `AnomalyDetectionEncoder` **apenas** no X_train (evita data leakage)
6. Aplica `SimpleImputer(fill_value=0)` nos NaN residuais (semanticamente: ausente = não se aplica = 0)

In [ ]:
def preprocessar_dataset(nome: str, arquivo_parquet: str, labels_df: pd.DataFrame) -> dict:
    """
    Carrega, une com labels, amostra, divide e codifica um dataset.

    Retorna um dicionário com:
      X_train, X_test, y_train, y_test, feature_names,
      encoder, imputer, df_test (com KEY_COLS para rastreabilidade)
    """
    print(f"\n{'='*60}")
    print(f"Dataset: {nome}")
    print(f"{'='*60}")

    # 1. Carregar parquet
    df = pd.read_parquet(arquivo_parquet)
    print(f"  Shape original: {df.shape}")

    # 2. Garantir KEY_COLS como string no df principal
    for col in KEY_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str)
        else:
            raise KeyError(f"Coluna '{col}' não encontrada no dataset '{nome}'. "
                           f"Colunas disponíveis: {list(df.columns[:10])} ...")

    # 3. Merge com ground truth (left join → NaN → 0)
    df = df.merge(
        labels_df[KEY_COLS + ['is_anomaly']],
        on=KEY_COLS,
        how='left'
    )
    df['is_anomaly'] = df['is_anomaly'].fillna(0).astype(int)
    n_positivos = df['is_anomaly'].sum()
    print(f"  Após merge — anomalias: {n_positivos:,} ({n_positivos/len(df):.2%})")

    # 4. Amostragem estratificada
    n_amostras = min(SAMPLE_SIZE, len(df))
    if n_amostras < len(df):
        df = df.groupby('is_anomaly', group_keys=False).apply(
            lambda g: g.sample(
                n=max(1, round(n_amostras * len(g) / len(df))),
                random_state=RANDOM_STATE
            )
        ).reset_index(drop=True)
        print(f"  Após amostragem: {len(df):,} registros "
              f"(anomalias: {df['is_anomaly'].sum():,} = {df['is_anomaly'].mean():.2%})")

    # 5. Separar features, labels e chave
    y = df['is_anomaly'].values
    key_df = df[KEY_COLS].copy()
    X = df.drop(columns=KEY_COLS + ['is_anomaly'], errors='ignore')

    # 6. Split treino/teste estratificado 80/20
    X_train, X_test, y_train, y_test, key_train, key_test = train_test_split(
        X, y, key_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )
    print(f"  Treino: {X_train.shape} | Teste: {X_test.shape}")

    # 7. Encoder — ajustado APENAS no treino
    encoder = AnomalyDetectionEncoder(key_cols=[])   # KEY_COLS já removidas
    encoder.fit(X_train)
    X_train_enc = encoder.transform(X_train)
    X_test_enc  = encoder.transform(X_test)

    # 8. Imputação residual de NaN (fill_value=0: ausente = não se aplica)
    imputer = SimpleImputer(strategy='constant', fill_value=0)
    X_train_enc = imputer.fit_transform(X_train_enc)
    X_test_enc  = imputer.transform(X_test_enc)

    feature_names = encoder.get_feature_info()['feature'].tolist() \
        if hasattr(encoder, 'get_feature_info') else list(X_train.columns)

    print(f"  Shape final (treino): {X_train_enc.shape}")
    print(f"  NaN residuais (teste): {np.isnan(X_test_enc).sum()}")

    return {
        'X_train':      X_train_enc,
        'X_test':       X_test_enc,
        'y_train':      y_train,
        'y_test':       pd.Series(y_test, name='is_anomaly'),
        'key_test':     key_test.reset_index(drop=True),
        'feature_names': feature_names,
        'encoder':      encoder,
        'imputer':      imputer,
    }


# ─────────────────────────────────────────────────────────────
# Processar os 3 datasets
# ─────────────────────────────────────────────────────────────
dataset_files = {
    'estabel': os.path.join(DATA_DIR, 'df_estabel_final.parquet'),
    'lav_temp': os.path.join(DATA_DIR, 'df_lav_temp_final.parquet'),
    'pecu':    os.path.join(DATA_DIR, 'df_pecu_final.parquet'),
}

datasets = {}
for nome, arquivo in dataset_files.items():
    if not os.path.exists(arquivo):
        print(f"[AVISO] Arquivo não encontrado, pulando dataset '{nome}': {arquivo}")
        continue
    datasets[nome] = preprocessar_dataset(nome, arquivo, labels_df)

print(f"\n{'='*60}")
print(f"Datasets processados: {list(datasets.keys())}")

## Seção 4 — Registro de Modelos (MODELS_REGISTRY)

Todos os modelos são instanciados via **factories** (lambdas) para garantir que cada execução crie um objeto fresco, sem estado compartilhado entre datasets.

### Como adicionar um novo modelo

```python
MODELS_REGISTRY["MeuModelo"] = {
    "factory":   lambda: MinhaClasse(contamination=CONTAMINATION, ...),
    "family":    "Ensemble",   # Ensemble | Probabilistic | Proximity | LinearProjection
    "rationale": "Breve justificativa da escolha",
}
```

> **Famílias** influenciam a estratégia de explicabilidade na Seção 9:
> - `Ensemble` + IForest → `shap.TreeExplainer`
> - `Probabilistic` (ECOD/COPOD) → atributos internos de score por feature
> - demais → `shap.KernelExplainer` (amostra de background)

In [ ]:
MODELS_REGISTRY = {
    "IForest": {
        "factory": lambda: IForest(
            contamination=CONTAMINATION,
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "family": "Ensemble",
        "rationale": (
            "Isolation Forest cria partições aleatórias do espaço de features; "
            "anomalias são isoladas com menos divisões. Robusto a dados esparsos e "
            "de alta dimensionalidade, sem assumir distribuição paramétrica."
        ),
    },
    "HBOS": {
        "factory": lambda: HBOS(
            contamination=CONTAMINATION,
            n_bins=50,
        ),
        "family": "Histograma",
        "rationale": (
            "Histogram-Based Outlier Score calcula a densidade de cada feature "
            "independentemente via histogramas. Extremamente rápido e eficiente "
            "em dados mistos com muitas features binárias/discretas."
        ),
    },
    "ECOD": {
        "factory": lambda: ECOD(
            contamination=CONTAMINATION,
            n_jobs=-1
        ),
        "family": "Probabilistic",
        "rationale": (
            "Empirical Cumulative Distribution-based Outlier Detection usa a "
            "distribuição empírica cumulativa de cada feature. Sem parâmetros "
            "críticos para ajuste; eficaz em dados não-paramétricos."
        ),
    },
    "COPOD": {
        "factory": lambda: COPOD(
            contamination=CONTAMINATION,
            n_jobs=-1
        ),
        "family": "Probabilistic",
        "rationale": (
            "Copula-based Outlier Detection usa cópulas empíricas para modelar "
            "dependências entre features. Boa calibração de scores e complementar ao ECOD."
        ),
    },
    "LODA": {
        "factory": lambda: LODA(
            contamination=CONTAMINATION,
            n_bins=50,
        ),
        "family": "Ensemble",
        "rationale": (
            "Lightweight Online Detector of Anomalies usa projeções aleatórias esparsas "
            "em histogramas 1D. Especialmente adequado para dados esparsos e alta "
            "dimensionalidade — cada projeção envolve poucas features originais."
        ),
    },
    "LOF": {
        "factory": lambda: LOF(
            contamination=CONTAMINATION,
            n_neighbors=20,
            n_jobs=-1
        ),
        "family": "Proximity",
        "rationale": (
            "Local Outlier Factor detecta anomalias por comparação de densidades locais. "
            "Captura anomalias contextuais que modelos globais tendem a perder; "
            "complementa os modelos de ensemble."
        ),
    },
}

print("MODELS_REGISTRY criado com os modelos:")
for name, cfg in MODELS_REGISTRY.items():
    print(f"  [{cfg['family']:15s}] {name}: {cfg['rationale'][:60]}...")

## Seção 4b — Experimento: Redução de Dimensionalidade

Avalia o impacto de aplicar **PCA** como pré-processamento antes do IForest, usando o dataset `estabel` como referência.

**Caveat sobre PCA em dados do censo:**
> O PCA preserva componentes de **alta variância**, mas anomalias em dados do censo podem se manifestar justamente nas features de **baixa variância** (ex.: combinações raras de variáveis binárias). Ao descartar componentes menores, o PCA pode suprimir o sinal de anomalia. Este experimento verifica empiricamente se o PCA ajuda ou prejudica.

**KPCA como detector direto:** O PyOD oferece `KPCA` como modelo de detecção — usa o erro de reconstrução via kernel PCA como score de anomalia, sem os riscos do PCA como pré-processamento.

In [ ]:
def _avaliar_dimred(nome_exp, X_train, X_test, y_test, contamination=CONTAMINATION):
    """Treina IForest com pré-processamento opcional e retorna AP e ROC-AUC."""
    if 'PCA' in nome_exp:
        pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
        X_tr  = pca.fit_transform(X_train)
        X_te  = pca.transform(X_test)
        n_comp = X_tr.shape[1]
        print(f"  PCA: {X_train.shape[1]} → {n_comp} componentes (95% variância)")
    else:
        X_tr, X_te = X_train, X_test

    model = IForest(contamination=contamination, n_estimators=100,
                    random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_tr)
    scores = model.decision_function(X_te)

    ap  = average_precision_score(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    print(f"  {nome_exp:<30s}  AP={ap:.4f}  ROC-AUC={auc:.4f}")
    return {'AP': ap, 'ROC-AUC': auc, 'model': model, 'scores': scores}


def _avaliar_kpca(X_train, X_test, y_test, contamination=CONTAMINATION):
    """Avalia KPCA como detector direto (PyOD)."""
    model = KPCA(contamination=contamination)
    model.fit(X_train)
    scores = model.decision_function(X_test)
    ap  = average_precision_score(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    print(f"  {'KPCA (detector direto)':<30s}  AP={ap:.4f}  ROC-AUC={auc:.4f}")
    return {'AP': ap, 'ROC-AUC': auc}


# ─────────────────────────────────────────────────────────────
# Executar experimento no dataset 'estabel'
# ─────────────────────────────────────────────────────────────
dimred_results = {}

if 'estabel' in datasets:
    ds = datasets['estabel']
    print("Experimento de redução de dimensionalidade — dataset: estabel\n")

    dimred_results['IForest (sem redução)'] = _avaliar_dimred(
        'IForest (sem redução)', ds['X_train'], ds['X_test'], ds['y_test'])

    dimred_results['IForest + PCA (95%)'] = _avaliar_dimred(
        'IForest + PCA (95%)', ds['X_train'], ds['X_test'], ds['y_test'])

    dimred_results['KPCA (detector direto)'] = _avaliar_kpca(
        ds['X_train'], ds['X_test'], ds['y_test'])

    # Resumo visual
    dimred_df = pd.DataFrame(dimred_results).T[['AP', 'ROC-AUC']].astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    dimred_df['AP'].plot(kind='bar', ax=axes[0], color=['steelblue','tomato','seagreen'], rot=15)
    axes[0].set_title('Average Precision')
    axes[0].set_ylim(0, 1)
    for bar in axes[0].patches:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    dimred_df['ROC-AUC'].plot(kind='bar', ax=axes[1], color=['steelblue','tomato','seagreen'], rot=15)
    axes[1].set_title('ROC-AUC')
    axes[1].set_ylim(0, 1)
    for bar in axes[1].patches:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle('Impacto da Redução de Dimensionalidade — IForest (estabel)', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Recomendação automática
    melhor = dimred_df['AP'].idxmax()
    print(f"\n→ Recomendação: '{melhor}' apresentou melhor AP ({dimred_df.loc[melhor, 'AP']:.4f}).")
    if 'PCA' in melhor:
        print("  O PCA melhorou o desempenho — considere incluir IForest+PCA no MODELS_REGISTRY.")
    else:
        print("  O PCA não trouxe ganho — prosseguir sem redução de dimensionalidade é adequado.")
else:
    print("[AVISO] Dataset 'estabel' não disponível. Pulando experimento de redução de dimensionalidade.")

## Seção 5 — Treinamento e Scoring

Treina cada modelo do `MODELS_REGISTRY` em cada dataset disponível. Os modelos são instanciados via factory a cada execução para garantir independência entre datasets.

In [ ]:
def treinar_e_avaliar(model_name: str, model, X_train, X_test):
    """
    Treina o modelo e retorna scores, predições, tempo e o modelo ajustado.
    O modelo ajustado é armazenado para uso posterior na explicabilidade.
    """
    t0 = time.time()
    model.fit(X_train)
    scores = model.decision_function(X_test)
    preds  = model.predict(X_test)
    elapsed = time.time() - t0
    return scores, preds, elapsed, model   # retorna o modelo ajustado


# ─────────────────────────────────────────────────────────────
# Loop principal: treino + scoring em todos datasets × modelos
# ─────────────────────────────────────────────────────────────
# Estrutura: all_results[dataset_name][model_name] = {scores, preds, time, fitted_model}
all_results = {}

for ds_name, ds in datasets.items():
    print(f"\n{'─'*55}")
    print(f"Dataset: {ds_name}  (X_test shape: {ds['X_test'].shape})")
    print(f"{'─'*55}")
    all_results[ds_name] = {}

    for model_name, model_cfg in MODELS_REGISTRY.items():
        model = model_cfg['factory']()
        scores, preds, elapsed, fitted = treinar_e_avaliar(
            model_name, model, ds['X_train'], ds['X_test']
        )
        all_results[ds_name][model_name] = {
            'scores':       scores,
            'preds':        preds,
            'time':         elapsed,
            'fitted_model': fitted,
        }
        n_flagged = (preds == 1).sum()
        print(f"  {model_name:<10s}  tempo={elapsed:5.1f}s  "
              f"flagged={n_flagged:4d} ({n_flagged/len(preds):.2%})")

print("\nTreinamento concluído.")

## Seção 6 — Métricas de Avaliação

Métricas calculadas para cada modelo × dataset:

| Métrica | Descrição |
|---------|-----------|
| **Average Precision (AP)** | Métrica principal — área sob a curva P×R; penaliza modelos que posicionam FPs antes de TPs |
| **ROC-AUC** | Capacidade discriminativa global |
| **Precision@K** | Precisão nos top-K scores onde K = `⌊N × contamination⌋` |
| **F1@contamination** | F1 usando threshold no percentil `(1 - contamination)` |

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve


def precision_at_k(y_true, scores, k: int) -> float:
    """Precisão nos top-K scores mais altos."""
    top_k_idx = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top_k_idx].mean())


def calcular_metricas(y_true, scores, contamination: float = CONTAMINATION) -> dict:
    """Calcula AP, ROC-AUC, P@K e F1@contamination."""
    y = np.asarray(y_true)
    k = max(1, math.floor(len(y) * contamination))
    threshold = np.percentile(scores, (1 - contamination) * 100)
    preds_at_cont = (scores >= threshold).astype(int)

    tp = int(((preds_at_cont == 1) & (y == 1)).sum())
    fp = int(((preds_at_cont == 1) & (y == 0)).sum())
    fn = int(((preds_at_cont == 0) & (y == 1)).sum())
    denom = 2 * tp + fp + fn
    f1 = (2 * tp / denom) if denom > 0 else 0.0

    return {
        'avg_precision':  average_precision_score(y, scores),
        'roc_auc':        roc_auc_score(y, scores),
        'precision_at_k': precision_at_k(y, scores, k),
        'f1_at_cont':     f1,
        'k':              k,
    }


# ─────────────────────────────────────────────────────────────
# Calcular métricas para todos os experimentos
# ─────────────────────────────────────────────────────────────
metrics_rows = []
for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    for model_name, res in ds_results.items():
        m = calcular_metricas(y_test, res['scores'])
        metrics_rows.append({
            'dataset':        ds_name,
            'model':          model_name,
            'family':         MODELS_REGISTRY[model_name]['family'],
            'avg_precision':  m['avg_precision'],
            'roc_auc':        m['roc_auc'],
            'precision_at_k': m['precision_at_k'],
            'f1_at_cont':     m['f1_at_cont'],
            'time_s':         all_results[ds_name][model_name]['time'],
        })

ranking_df = pd.DataFrame(metrics_rows).sort_values(
    ['dataset', 'avg_precision'], ascending=[True, False]
)

print("Ranking por Average Precision:\n")
print(ranking_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
def _plot_curvas(ds_name, ds_results, y_test):
    """Plota curvas ROC e P×R para todos os modelos de um dataset."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    palette = sns.color_palette('tab10', n_colors=len(ds_results))

    for (model_name, res), color in zip(ds_results.items(), palette):
        scores = res['scores']
        # ROC
        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_val = roc_auc_score(y_test, scores)
        axes[0].plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})", color=color)
        # PR
        prec, rec, _ = precision_recall_curve(y_test, scores)
        ap_val = average_precision_score(y_test, scores)
        axes[1].plot(rec, prec, label=f"{model_name} (AP={ap_val:.3f})", color=color)

    # Baseline aleatório
    pos_rate = np.asarray(y_test).mean()
    axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random')
    axes[1].axhline(pos_rate, color='k', ls='--', lw=0.8,
                    label=f'Random (AP={pos_rate:.3f})')

    axes[0].set(title=f'Curva ROC — {ds_name}', xlabel='FPR', ylabel='TPR')
    axes[1].set(title=f'Curva PR — {ds_name}',  xlabel='Recall', ylabel='Precision')
    for ax in axes:
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


def _plot_score_hist(ds_name, ds_results, y_test):
    """Histograma dos scores de anomalia por classe real."""
    n_models = len(ds_results)
    cols = min(3, n_models)
    rows = math.ceil(n_models / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten()

    for idx, (model_name, res) in enumerate(ds_results.items()):
        scores = res['scores']
        ax = axes[idx]
        for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Anomalia')]:
            mask = np.asarray(y_test) == label
            ax.hist(scores[mask], bins=40, alpha=0.6, color=color,
                    label=f'{name} (n={mask.sum()})', density=True)
        ax.set_title(f'{model_name}', fontsize=10)
        ax.set_xlabel('Score'); ax.legend(fontsize=8)

    for ax in axes[n_models:]:
        ax.set_visible(False)

    plt.suptitle(f'Distribuição de Scores por Classe — {ds_name}', fontsize=12)
    plt.tight_layout()
    plt.show()


# ─────────────────────────────────────────────────────────────
# Gerar plots para cada dataset
# ─────────────────────────────────────────────────────────────
for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    _plot_curvas(ds_name, ds_results, y_test)
    _plot_score_hist(ds_name, ds_results, y_test)

## Seção 7 — Análise de Concordância entre Modelos

Avalia quanto os modelos concordam entre si:
1. **Mapa de correlação de Spearman** dos scores (quanto os modelos ranqueiam os registros de forma similar)
2. **Contagem de consenso**: quantos modelos sinalizam cada registro como anomalia
3. **Top-20 por consenso**: registros mais suspeitos segundo o maior número de modelos

In [ ]:
from scipy.stats import spearmanr

concordance_summary = {}

for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    key_test = datasets[ds_name]['key_test']
    model_names = list(ds_results.keys())
    n = len(model_names)

    # ── 1. Correlação de Spearman entre scores ──────────────────
    scores_matrix = np.column_stack([ds_results[m]['scores'] for m in model_names])
    corr_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            corr_matrix[i, j] = spearmanr(scores_matrix[:, i], scores_matrix[:, j]).statistic

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        pd.DataFrame(corr_matrix, index=model_names, columns=model_names),
        annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
        ax=axes[0], square=True
    )
    axes[0].set_title(f'Correlação de Spearman (scores) — {ds_name}')

    # ── 2. Consenso: quantos modelos sinalizam cada registro ────
    preds_matrix = np.column_stack([ds_results[m]['preds'] for m in model_names])
    consensus = preds_matrix.sum(axis=1)   # número de modelos que sinalizaram 1

    consensus_df = key_test.copy()
    consensus_df['consensus_count'] = consensus
    consensus_df['is_anomaly']      = y_test.values
    concordance_summary[ds_name]    = consensus_df

    # Barplot consenso
    max_count = int(consensus.max()) + 1
    labels_order = list(range(max_count))
    counts = pd.Series(consensus).value_counts().reindex(labels_order, fill_value=0)
    counts.plot(kind='bar', ax=axes[1], color='steelblue', rot=0)
    axes[1].set_xlabel('Nº de modelos que sinalizaram anomalia')
    axes[1].set_ylabel('Nº de registros')
    axes[1].set_title(f'Histograma de Consenso — {ds_name}')
    for bar in axes[1].patches:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── 3. Top-20 por consenso ──────────────────────────────────
    print(f"\nTop-20 registros por consenso — {ds_name}:")
    top20 = (
        consensus_df
        .sort_values('consensus_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
    print(top20.to_string())

## Seção 8 — Avaliação Combinada Multi-Dataset

Estratégia: para cada estabelecimento, o **score final** é o **máximo** entre os scores dos 3 datasets (estabel, lav_temp, pecu). A justificativa é que um estabelecimento anômalo em *qualquer* dimensão (estrutural, vegetal ou pecuária) deve ser sinalizado.

- Scores dos datasets `lav_temp` e `pecu` são opcionais (NaN se o estabelecimento não pertence ao dataset)
- `combined_score = max(score_estabel, score_lav_temp, score_pecu)` ignorando NaN

> **Nota:** a avaliação combinada usa apenas os registros com `y_true` disponível (provenientes do arquivo de labels).

In [ ]:
def _build_score_df(ds_name: str, model_name: str) -> pd.DataFrame:
    """Constrói DataFrame com KEY_COLS + score + y_true para um dado dataset/modelo."""
    ds = datasets[ds_name]
    scores = all_results[ds_name][model_name]['scores']
    df_out = ds['key_test'].copy()
    df_out[f'score_{ds_name}'] = scores
    df_out['y_true'] = ds['y_test'].values
    return df_out


# ─────────────────────────────────────────────────────────────
# Para cada modelo, construir avaliação combinada
# ─────────────────────────────────────────────────────────────
combined_metrics_rows = []
combined_dfs = {}   # {model_name: DataFrame com scores combinados}

for model_name in MODELS_REGISTRY.keys():
    parts = []
    for ds_name in all_results:
        if model_name in all_results[ds_name]:
            parts.append(_build_score_df(ds_name, model_name))

    if not parts:
        continue

    # Merge externo por KEY_COLS — cada dataset contribui com sua coluna de score
    merged = parts[0]
    for part in parts[1:]:
        # Remove y_true duplicado antes do merge; será preservado do primeiro
        part_no_ytrue = part.drop(columns=['y_true'])
        merged = merged.merge(part_no_ytrue, on=KEY_COLS, how='outer')
    # Recuperar y_true para registros vindos apenas de datasets secundários
    # (garantia: qualquer registro com y_true=1 está em labels_df, logo em estabel)

    score_cols = [c for c in merged.columns if c.startswith('score_')]
    merged['combined_score'] = merged[score_cols].max(axis=1)
    merged['y_true'] = merged['y_true'].fillna(0).astype(int)
    combined_dfs[model_name] = merged

    # Métricas combinadas
    valid = merged.dropna(subset=['combined_score'])
    if valid['y_true'].nunique() < 2:
        continue
    m = calcular_metricas(valid['y_true'], valid['combined_score'])
    combined_metrics_rows.append({
        'model':            model_name,
        'avg_precision':    m['avg_precision'],
        'roc_auc':          m['roc_auc'],
        'precision_at_k':   m['precision_at_k'],
        'f1_at_cont':       m['f1_at_cont'],
    })

combined_metrics_df = pd.DataFrame(combined_metrics_rows).sort_values(
    'avg_precision', ascending=False
)

print("Métricas Combinadas (max-score multi-dataset):\n")
print(combined_metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ─────────────────────────────────────────────────────────────
# Comparação: individual vs combinado (melhor dataset individual)
# ─────────────────────────────────────────────────────────────
if not combined_metrics_df.empty:
    # Para cada modelo, pegar o melhor AP individual (entre os datasets)
    individual_best = (
        ranking_df.groupby('model')['avg_precision'].max().reset_index()
        .rename(columns={'avg_precision': 'AP_individual_best'})
    )
    comp = combined_metrics_df[['model', 'avg_precision']].rename(
        columns={'avg_precision': 'AP_combined'}
    ).merge(individual_best, on='model')

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(comp))
    width = 0.35
    bars1 = ax.bar(x - width/2, comp['AP_individual_best'], width, label='Melhor individual', color='steelblue')
    bars2 = ax.bar(x + width/2, comp['AP_combined'],        width, label='Combinado (max)',  color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(comp['model'], rotation=20, ha='right')
    ax.set_ylabel('Average Precision')
    ax.set_title('AP: Melhor Dataset Individual vs. Score Combinado')
    ax.legend()
    ax.set_ylim(0, 1)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Scatter: score_estabel × score_lav_temp (se ambos disponíveis)
# colorido por y_true — permite visualizar onde os datasets concordam
# ─────────────────────────────────────────────────────────────
best_model_combined = (
    combined_metrics_df.iloc[0]['model']
    if not combined_metrics_df.empty else None
)

if best_model_combined and best_model_combined in combined_dfs:
    df_scatter = combined_dfs[best_model_combined]
    has_estabel  = 'score_estabel'  in df_scatter.columns
    has_lav_temp = 'score_lav_temp' in df_scatter.columns

    if has_estabel and has_lav_temp:
        sub = df_scatter.dropna(subset=['score_estabel', 'score_lav_temp'])
        fig, ax = plt.subplots(figsize=(7, 5))
        colors = sub['y_true'].map({0: 'steelblue', 1: 'tomato'})
        sc = ax.scatter(sub['score_estabel'], sub['score_lav_temp'],
                        c=colors, alpha=0.5, s=20)
        from matplotlib.patches import Patch
        legend_els = [Patch(facecolor='steelblue', label='Normal'),
                      Patch(facecolor='tomato',    label='Anomalia')]
        ax.legend(handles=legend_els)
        ax.set_xlabel('Score estabel')
        ax.set_ylabel('Score lav_temp')
        ax.set_title(f'Score estabel × lav_temp — {best_model_combined}')
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] Scatter omitido: um dos datasets (estabel/lav_temp) não está disponível.")

## Seção 9 — Explicabilidade

A explicabilidade é realizada sobre o **melhor modelo por Average Precision** (ou o modelo configurado em `EXPLAIN_MODEL_NAME`), no dataset `EXPLAIN_DATASET`.

### Estratégia por família de modelo

| Família | Abordagem |
|---------|-----------|
| `Ensemble` (IForest) | `shap.TreeExplainer` — rápido, exato para árvores de decisão |
| `Probabilistic` (ECOD/COPOD) | Atributos internos por feature (`score_left_`, `score_right_`) + KernelExplainer como fallback |
| `Histograma` (HBOS) | `shap.KernelExplainer` com amostra de background |
| demais | `shap.KernelExplainer` |

### Outputs gerados
1. **Importância global** — barplot dos top-20 features por `mean(|SHAP|)`
2. **Waterfall local** — explicação dos top-5 anomalias (maior score)
3. **Distribuição** — violin plot comparando valores SHAP de anomalias vs normais nas top-10 features

> **Para explorar outro modelo:** altere `EXPLAIN_MODEL_NAME` na Seção 0 (ou na célula abaixo) e reexecute esta seção.

In [ ]:
# ── Configuração da explicabilidade ────────────────────────────
# Altere as variáveis abaixo para explorar outro modelo/dataset
_explain_model  = EXPLAIN_MODEL_NAME   # None = auto
_explain_ds     = EXPLAIN_DATASET      # 'estabel' | 'lav_temp' | 'pecu'

# ── Seleção automática do melhor modelo por AP ──────────────────
if _explain_model is None:
    _explain_model = (
        ranking_df[ranking_df['dataset'] == _explain_ds]
        .sort_values('avg_precision', ascending=False)
        .iloc[0]['model']
    )
    print(f"Melhor modelo por AP no dataset '{_explain_ds}': {_explain_model}")
else:
    print(f"Modelo selecionado manualmente: {_explain_model}")

# ── Dados para explicabilidade ──────────────────────────────────
ds_exp   = datasets[_explain_ds]
X_test   = ds_exp['X_test']
y_test   = ds_exp['y_test'].values
feat_names = ds_exp['feature_names']
scores   = all_results[_explain_ds][_explain_model]['scores']
fitted   = all_results[_explain_ds][_explain_model]['fitted_model']
family   = MODELS_REGISTRY[_explain_model]['family']

# Limitar para 500 amostras de teste para legibilidade/velocidade
N_EXPLAIN = min(500, len(X_test))
rng = np.random.default_rng(RANDOM_STATE)
exp_idx = rng.choice(len(X_test), size=N_EXPLAIN, replace=False)
X_exp   = X_test[exp_idx]
y_exp   = y_test[exp_idx]
scores_exp = scores[exp_idx]

print(f"\nFamília: {family} | Amostras para explicabilidade: {N_EXPLAIN}")
print(f"Anomalias na amostra: {y_exp.sum()} ({y_exp.mean():.2%})")

In [ ]:
def _get_shap_values(fitted_model, X_explain, family: str, feat_names: list):
    """
    Calcula SHAP values de acordo com a família do modelo.
    Retorna (shap_values_array, explainer).
    """
    model_class = type(fitted_model).__name__

    if family == 'Ensemble' and 'IForest' in model_class:
        # TreeExplainer: usa as árvores internas diretamente
        # fitted_model.detector_ é o sklearn IsolationForest subjacente
        try:
            inner = fitted_model.detector_
        except AttributeError:
            inner = fitted_model
        explainer = shap.TreeExplainer(inner)
        shap_vals = explainer.shap_values(X_explain)

    elif family == 'Probabilistic' and model_class in ('ECOD', 'COPOD'):
        # ECOD/COPOD armazenam scores por feature
        # Usamos a decomposição interna como proxy de importância,
        # e KernelExplainer para SHAP values propriamente ditos
        # Limitamos background a 100 amostras para velocidade
        background = shap.sample(X_explain, min(100, len(X_explain)),
                                 random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(
            fitted_model.decision_function, background
        )
        shap_vals = explainer.shap_values(X_explain[:50])  # KernelSHAP é lento
        X_explain = X_explain[:50]

    else:
        # Fallback: KernelExplainer para qualquer outro modelo
        background = shap.sample(X_explain, min(100, len(X_explain)),
                                 random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(
            fitted_model.decision_function, background
        )
        shap_vals = explainer.shap_values(X_explain[:50])
        X_explain = X_explain[:50]

    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]

    return shap_vals, explainer, X_explain


print("Calculando SHAP values (pode demorar alguns minutos para KernelExplainer)...")
shap_values, _explainer, X_explain_final = _get_shap_values(
    fitted, X_exp, family, feat_names
)
print(f"SHAP values calculados: shape = {shap_values.shape}")

In [ ]:
# ── Plot 1: Importância Global (mean |SHAP|) ───────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feat_imp = pd.DataFrame({
    'feature':    feat_names[:len(mean_abs_shap)],
    'importance': mean_abs_shap
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title(f'Top-20 Features por Importância Global (mean |SHAP|)\n{_explain_model} — {_explain_ds}')
ax.set_xlabel('mean(|SHAP value|)')
plt.tight_layout()
plt.show()

# ── Plot 2: SHAP Summary (beeswarm) ─────────────────────────────
shap_exp = shap.Explanation(
    values=shap_values,
    data=X_explain_final,
    feature_names=feat_names[:shap_values.shape[1]]
)
plt.figure(figsize=(9, 6))
shap.plots.beeswarm(shap_exp, max_display=20, show=False)
plt.title(f'SHAP Beeswarm — {_explain_model} ({_explain_ds})')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Waterfall para top-5 anomalias ─────────────────────
# Usa scores da amostra de explicabilidade (alinhado com shap_values)
n_exp = shap_values.shape[0]
scores_aligned = scores_exp[:n_exp]
top5_idx = np.argsort(scores_aligned)[::-1][:5]

for rank, idx in enumerate(top5_idx, start=1):
    exp_single = shap.Explanation(
        values=shap_values[idx],
        base_values=_explainer.expected_value if hasattr(_explainer, 'expected_value') else 0.0,
        data=X_explain_final[idx],
        feature_names=feat_names[:shap_values.shape[1]]
    )
    plt.figure(figsize=(9, 4))
    shap.plots.waterfall(exp_single, max_display=15, show=False)
    plt.title(f'Waterfall — Top-{rank} anomalia | score={scores_aligned[idx]:.4f} | {_explain_model}')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Plot 4: Violin — distribuição SHAP anomalias vs normais ─────
n_exp   = shap_values.shape[0]
y_exp_aligned = y_exp[:n_exp]

# Top-10 features mais importantes
top10_features = feat_imp.head(10)['feature'].tolist()
top10_idx = [feat_names.index(f) for f in top10_features if f in feat_names]

if top10_idx:
    rows_data = []
    for i in range(n_exp):
        label = 'Anomalia' if y_exp_aligned[i] == 1 else 'Normal'
        for fi in top10_idx:
            rows_data.append({
                'feature': feat_names[fi],
                'shap_value': shap_values[i, fi],
                'classe': label,
            })
    violin_df = pd.DataFrame(rows_data)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.violinplot(
        data=violin_df, x='feature', y='shap_value', hue='classe',
        split=True, inner='quart', palette={'Normal': 'steelblue', 'Anomalia': 'tomato'},
        ax=ax
    )
    ax.set_title(f'Distribuição SHAP por Classe — Top-10 features\n{_explain_model} ({_explain_ds})')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.axhline(0, color='black', lw=0.8, ls='--')
    plt.tight_layout()
    plt.show()
else:
    print("[AVISO] Nenhuma das top features encontrada em feat_names para violin plot.")

## Seção 10 — Exportação de Resultados

Exporta para `OUTPUT_DIR`:
- `anomaly_scores_{dataset}.parquet` — scores de todos os modelos por registro (com KEY_COLS)
- `anomaly_scores_combined.parquet` — score combinado (max) com KEY_COLS e y_true
- `model_ranking.csv` — ranking de métricas individual e combinado

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Scores por dataset (todos os modelos) ────────────────────
for ds_name, ds_results in all_results.items():
    ds = datasets[ds_name]
    export_df = ds['key_test'].copy()
    export_df['y_true'] = ds['y_test'].values
    for model_name, res in ds_results.items():
        export_df[f'score_{model_name}'] = res['scores']
        export_df[f'pred_{model_name}']  = res['preds']

    out_path = os.path.join(OUTPUT_DIR, f'anomaly_scores_{ds_name}.parquet')
    export_df.to_parquet(out_path, index=False)
    print(f"Exportado: {out_path}  ({export_df.shape})")

# ── 2. Score combinado ──────────────────────────────────────────
if combined_dfs:
    # Usar o melhor modelo combinado
    best_cm = combined_metrics_df.iloc[0]['model'] if not combined_metrics_df.empty else list(combined_dfs.keys())[0]
    combined_export = combined_dfs[best_cm].copy()
    combined_export.attrs['best_model'] = best_cm
    out_combined = os.path.join(OUTPUT_DIR, 'anomaly_scores_combined.parquet')
    combined_export.to_parquet(out_combined, index=False)
    print(f"Exportado: {out_combined}  ({combined_export.shape})")

# ── 3. Ranking de métricas ──────────────────────────────────────
# Individual
ranking_export = ranking_df.copy()
ranking_export['scope'] = 'individual'
# Combinado
if not combined_metrics_df.empty:
    comb_exp = combined_metrics_df.copy()
    comb_exp['dataset'] = 'combined'
    comb_exp['scope'] = 'combined'
    comb_exp['family'] = comb_exp['model'].map(
        {k: v['family'] for k, v in MODELS_REGISTRY.items()}
    )
    ranking_export = pd.concat([ranking_export, comb_exp], ignore_index=True)

out_ranking = os.path.join(OUTPUT_DIR, 'model_ranking.csv')
ranking_export.to_csv(out_ranking, index=False)
print(f"Exportado: {out_ranking}  ({ranking_export.shape})")

print("\nExportação concluída.")
print(f"\nResumo final — melhor modelo por AP (dataset estabel):")
best_row = ranking_df[ranking_df['dataset'] == 'estabel'].iloc[0]
print(f"  Modelo:           {best_row['model']}")
print(f"  Average Precision:{best_row['avg_precision']:.4f}")
print(f"  ROC-AUC:          {best_row['roc_auc']:.4f}")
print(f"  Precision@K:      {best_row['precision_at_k']:.4f}")
print(f"  F1@contamination: {best_row['f1_at_cont']:.4f}")